In [ ]:
import sys
from pathlib import Path

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if root is None:
    raise SystemExit(f"Could not find src/config.py above {Path.cwd()}. Check that you saved config.py and osm.py inside the src folder.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Project root:", root)

%load_ext autoreload
%autoreload 2
import pandas as pd, geopandas as gpd
from src import config as C, osm
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

g = gpd.read_file(C.STUDY_GPKG).to_crs(C.CRS_METRIC)
if "area_km2" not in g.columns:
    g["area_km2"] = g.geometry.area / 1e6
print(len(g), "barangays |", g["Mun_City"].value_counts().to_dict())

In [ ]:
osm.ping()

for c in C.STUDY_CITIES:
    print(c, "vertices:", osm.n_vertices(osm.query_polygon(g, c)))
osm.cache_status()

In [ ]:
feat = osm.load_features(g)
print(len(feat), "features")

In [ ]:
log = pd.read_csv(C.OSM_DIR / "DOWNLOAD_LOG.txt", sep="\t", header=None,
                  names=["utc_time", "city", "key", "n", "osmnx", "server", "date_pin"])
print("Download window (UTC):", log["utc_time"].min(), "to", log["utc_time"].max())
log

In [ ]:
pts = osm.assign_types(feat, g)
print(len(pts), "features inside study barangays")
pts["osm_type"].value_counts()

In [ ]:
A = osm.audit(pts, g)
for name, tbl in A.items():
    print("\n==", name.upper()); display(tbl)

In [ ]:
osm.tag_checks(pts)

In [ ]:
rows = []
orig = C.MIN_OPEN_SPACE_AREA_M2
for thr in (200, 500, 1000):
    C.MIN_OPEN_SPACE_AREA_M2 = thr
    ft = pts.apply(osm.final_type, axis=1)
    s = ft.value_counts()
    s.name = thr
    rows.append(s)
C.MIN_OPEN_SPACE_AREA_M2 = orig
pd.DataFrame(rows).fillna(0).astype(int).rename_axis("min_area_m2")

In [ ]:
core = pts[pts["final_type"].notna()].copy()
per_brgy = (core.groupby("adm4_pcode").size()
            .reindex(g["adm4_pcode"], fill_value=0).rename("n_core"))
chk = g[["adm4_pcode", "Mun_City"]].merge(per_brgy, on="adm4_pcode")
print("Barangays with zero core third spaces, by city (count / share):")
z = chk.assign(zero=chk["n_core"] == 0).groupby("Mun_City")["zero"].agg(["sum", "mean"]).round(2)
z

In [ ]:
KEEP = ["element", "id", "element_type", "osmid", "name", "amenity", "leisure", "place",
        "access", "sport", "osm_type", "final_type", "geom_type", "area_m2", "named",
        "adm4_pcode", "Mun_City", "geometry"]
out = pts[[c for c in KEEP if c in pts.columns]]
C.INTERIM.mkdir(parents=True, exist_ok=True)
C.PROCESSED.mkdir(parents=True, exist_ok=True)
out.to_file(C.INTERIM / "third_space_candidates.gpkg", driver="GPKG")                 # all candidates, for audit
out[out["final_type"].notna()].to_file(C.PROCESSED / "third_spaces.gpkg", driver="GPKG")  # core set, metric CRS
print("saved", len(out), "candidates;", out["final_type"].notna().sum(), "core third spaces")